In [ ]:
# --- SPACEMAP notebook configuration ------------------------------------
# Every path below is built from DATA_DIR. Point it at your own data by
# setting SPACEMAP_DATA_DIR in your environment, or by editing the default.
import os
from pathlib import Path

DATA_DIR = os.environ.get("SPACEMAP_DATA_DIR", str(Path.home() / "spacemap-data"))
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print(f"Using DATA_DIR = {DATA_DIR}")


# Step 5: Batch Correction

This notebook performs batch correction on marker expression data from annotated cell segmentation results. It addresses technical variation between different experimental batches while preserving biological variation.

## Overview

The batch correction pipeline consists of:
1. **Data Discovery and Configuration** - Automatically detect available markers and configure analysis parameters
2. **Optional Metadata Integration** - Flexibly incorporate sample metadata (treatment, timepoints, etc.)
3. **Data Loading and Validation** - Load annotated CSV files with configurable batch assignment strategies
4. **Quality Control Checkpoint** - Pre-correction QC with histogram analysis and outlier detection
5. **Per-batch Standardization** - Z-score scaling within each batch
6. **SOM-based Clustering** - Group similar cells across batches
7. **Per-cluster Batch Correction** - Apply ComBat correction within each cluster
8. **Value Capping** - Constrain corrected values to original ranges
9. **Quality Control Visualization** - Generate before/after comparison plots

## Input Requirements
- **Required**: Annotated CSV files from Step 4 (tissue annotation pipeline)
- **Optional**: Metadata file with sample information (treatment, response, timepoints, etc.)
- Files should contain marker expression data and spatial coordinates

## Configuration Options
- **Marker Selection**: Automatically discover available markers or specify custom lists
- **Metadata Integration**: Optional metadata file with flexible batch_id assignment strategies
- **Quality Control**: Configurable outlier detection and filtering criteria
- **Batch Assignment**: Multiple strategies for assigning batch identifiers
- **Filtering Criteria**: Flexible cell filtering based on quality metrics and annotation status

## Output
- Combined corrected dataset with both raw and batch-corrected markers
- Individual corrected files for each sample
- Quality control visualizations (histograms, PCA, UMAP, density plots)
- Quality assessment reports and filtering statistics

## Key Features
- **Configurable for Different Datasets** - Adaptable to various marker panels and experimental designs
- **Optional Metadata Integration** - Flexible incorporation of experimental metadata
- **Comprehensive Quality Control** - Pre-correction outlier detection and post-correction validation
- **SOM-based Approach** - Clusters cells before correction to preserve biological structure
- **Per-cluster Correction** - More precise than global correction
- **Publication-Ready Output** - Clean, well-documented results suitable for publication

In [ ]:
import os
# Fix for protobuf version compatibility with TensorFlow/UMAP
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add src to path for imports
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

from spacemap.config import default_config
from spacemap.utils import log_time, check_file_exists

# Import SOM clustering
try:
    from minisom import MiniSom
    print("MiniSom imported successfully")
except ImportError:
    print("Warning: MiniSom not found. Installing...")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "minisom"], check=True)
    from minisom import MiniSom
    print("MiniSom installed and imported successfully")

# Import all batch correction utilities
from spacemap.batch_correction_utils import (
    # Core processing functions
    BatchCorrector, 
    extract_batch_id_from_filename,
    assign_universal_cell_ids,
    preserve_filtered_cells_in_output,
    # Data discovery and loading
    discover_data_and_markers,
    # Quality control
    perform_quality_control_analysis,
    detect_overcorrected_cells,
    apply_post_correction_qc_filters,
    # Configuration and validation
    apply_marker_selection_strategy,
    apply_data_filters,
    validate_markers_availability,
    # File operations
    save_batch_corrected_files,
    verify_cell_id_integrity
)
print("All batch correction utilities imported successfully")

# Set up logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Step 5: Batch Correction - Setup Complete")
print(f"Project root: {project_root}")
print(f"Configuration loaded")
print("Utilities available: True")

## Data Discovery and Initial Configuration

First, let's discover what data we have available and configure the analysis parameters.

In [ ]:
# =============================================================================
# DATA DISCOVERY: FIND AVAILABLE CSV FILES AND DISCOVER COLUMNS
# =============================================================================

print("Discovering available data files and columns...")

# Set initial paths
project_root = Path().resolve().parent
#INPUT_DIR = project_root / "outputs" / "annotated_csv"
INPUT_DIR = Path(f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv')
# Try to use utility function for data discovery
try:
    # Reload module to pick up changes
    import importlib
    from spacemap import batch_correction_utils
    importlib.reload(batch_correction_utils)
    from spacemap.batch_correction_utils import discover_data_and_markers
    

    discovery_results = discover_data_and_markers(INPUT_DIR, exclude_patterns=[])
    csv_files = discovery_results['csv_files']
    potential_markers = discovery_results['potential_markers']
    annotation_columns = discovery_results['annotation_columns']
    spatial_columns = discovery_results['spatial_columns']
    file_info = discovery_results['file_info']
    
    print("Data discovery completed using utility function (NO EXCLUSIONS APPLIED)")
    
except ImportError:
    print("Using fallback data discovery in notebook...")
    
    # Find all CSV files
    csv_files = list(INPUT_DIR.glob("*_fully_annotated.csv"))
    if not csv_files:
        csv_files = list(INPUT_DIR.glob("*.csv"))

    if not csv_files:
        print(f"No CSV files found in {INPUT_DIR}")
        print("Please check your input directory path.")
        potential_markers = []
        annotation_columns = []
        spatial_columns = []
        file_info = []
    else:
        print(f"Found {len(csv_files)} CSV files:")
        
        # Display files with their potential batch identifiers
        file_info = []
        for file in csv_files:
            filename = file.name
            filename_no_ext = filename[:-4] if filename.endswith('.csv') else filename
            
            # Try different extraction strategies
            if '.ome_cell_features_annotated' in filename_no_ext:
                batch_id = filename_no_ext.split('.ome_cell_features_annotated')[0]
            elif '_fully_annotated' in filename_no_ext:
                batch_id = filename_no_ext.split('_fully_annotated')[0]
            else:
                batch_id = filename_no_ext
                
            file_info.append({'file': filename, 'suggested_batch_id': batch_id})
            print(f"  {filename} -> suggested batch_id: '{batch_id}'")
        
        # Load first file to discover available columns
        print(f"\nAnalyzing columns from first file: {csv_files[0].name}")
        sample_df = pd.read_csv(csv_files[0], nrows=1000, low_memory=False)  # Load sample for speed
        
        all_columns = list(sample_df.columns)
        print(f"Total columns found: {len(all_columns)}")
        
        # Identify potential marker columns (ALL numeric columns, NO EXCLUSIONS)
        exclude_patterns = []  # NO EXCLUSIONS - include ALL numeric columns
        potential_markers = []
        
        for col in all_columns:
            if sample_df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
                # NO exclusion check - include ALL numeric columns
                potential_markers.append(col)
        
        print(f"\nPotential marker columns ({len(potential_markers)}):")
        for i, marker in enumerate(potential_markers):
            print(f"  {i+1:2d}. {marker}")
        
        # Check for common annotation columns
        annotation_columns = [col for col in all_columns if any(term in col.lower() for term in ['tissue', 'annotation', 'region', 'class'])]
        print(f"\nAnnotation/filtering columns found ({len(annotation_columns)}):")
        for col in annotation_columns:
            unique_vals = sample_df[col].unique()[:10]  # Show first 10 unique values
            print(f"  {col}: {unique_vals}")
        
        # Check for spatial columns - be more specific to avoid false positives
        spatial_columns = [col for col in all_columns if any(term in col.lower() for term in ['_centroid', 'major_axis', 'minor_axis', 'eccentricity', 'solidity', 'extent', 'perimeter'])]
        print(f"\nSpatial/morphology columns found ({len(spatial_columns)}):")
        for col in spatial_columns:
            print(f"  {col}")

print(f"\nData discovery summary:")
print(f"  CSV files found: {len(csv_files)}")
print(f"  Potential markers: {len(potential_markers)}")
print(f"  Annotation columns: {len(annotation_columns)}")
print(f"  Spatial columns: {len(spatial_columns)}")

# ============================================================================= 
# COPY-PASTE FRIENDLY MARKER LIST 
# =============================================================================
print(f"\n" + "="*60)
print("COPY-PASTE FRIENDLY MARKER LIST")
print("="*60)
print("Copy the markers you want from the list below and paste them into MARKERS_TO_CORRECT:")
print("")

# Format markers for easy copy-paste
marker_strings = [f'"{marker}"' for marker in potential_markers]
markers_per_line = 5
for i in range(0, len(marker_strings), markers_per_line):
    line_markers = marker_strings[i:i+markers_per_line]
    if i + markers_per_line < len(marker_strings):
        print("    " + ", ".join(line_markers) + ",")
    else:
        print("    " + ", ".join(line_markers))

print(f"\nTotal available markers: {len(potential_markers)}")
print("Simply copy the ones you want and paste them into the MARKERS_TO_CORRECT list below.")

## STEP 1: Comprehensive Configuration & Setup

This step contains all configuration parameters for the entire batch correction pipeline:

**Core Settings:**
- Input/output paths and metadata configuration  
- Marker selection strategy and manual marker list
- Cell ID assignment and batch identification

**Quality Control Parameters:**
- Pre-correction QC: outlier detection, area filtering, histogram generation
- Post-correction QC: overcorrection thresholds, correction distance limits, UMAP outlier filtering

**Batch Correction Parameters:**
- SOM clustering parameters (grid size, iterations, random seed)
- Processing and visualization settings

All thresholds defined upfront for consistent processing throughout the pipeline.

In [ ]:
# =============================================================================
# BATCH CORRECTION CONFIGURATION - MODIFY FOR YOUR DATASET
# =============================================================================

# Input/Output paths
#INPUT_DIR = default_config.annotated_csv_dir
INPUT_DIR = Path(f'{DATA_DIR}/example_study/registration/roi/cellpose/annotated_csv')

OUTPUT_DIR = default_config.output_dir / "batch_corrected"
PLOTS_DIR = OUTPUT_DIR / "plots"
QC_DIR = OUTPUT_DIR / "quality_control"

# =============================================================================
# FILE AND BATCH ID CONFIGURATION (SIMPLIFIED)
# =============================================================================

# File patterns to search for CSV files (tried in order)
CSV_FILE_PATTERNS = [
    "*_fully_annotated.csv", 
    "*_annotated.csv", 
    "*.csv"
]

# Batch ID assignment strategy (choose one):
# 'metadata' - Use column from metadata file
# 'filename' - Extract from filename (removes common suffixes automatically)
# 'filename_custom' - Use custom regex pattern
# 'column'   - Use an existing column in the CSV files
BATCH_ID_STRATEGY = 'metadata'

# Column name in the CSV files containing the Batch ID (only if BATCH_ID_STRATEGY = 'column')
BATCH_COLUMN_NAME = 'batch_id'

# Suffixes to automatically remove when extracting batch_id from filename
# (used for 'filename' strategy - first matching suffix is removed)
FILENAME_SUFFIXES_TO_REMOVE = [
    '.ome_cell_features_annotated.csv',
    '.ome_cell_fully_annotated.csv', 
    '_fully_annotated.csv',
    '_annotated.csv',
    '.csv'
]

# Custom regex pattern for batch_id extraction (only for 'filename_custom' strategy)
BATCH_ID_CUSTOM_PATTERN = r'(P\d+_roi_\d+)'

# =============================================================================
# METADATA CONFIGURATION (OPTIONAL)
# =============================================================================

# Set to True if you have a metadata file
USE_METADATA = True
METADATA_FILE = default_config.data_dir / "cohort" / "slides_patients_arm_timepoint.csv"

# Metadata column mapping
METADATA_FILE_PREFIX_COLUMN = 'file_prefix'  # Column with file prefixes for matching
METADATA_BATCH_ID_COLUMN = 'batch_id'        # Column to use as batch_id (or None to create one using components)

# If METADATA_BATCH_ID_COLUMN is None, create batch_id from these columns
METADATA_BATCH_ID_COMPONENTS = ['patient', 'timepoint']
METADATA_BATCH_ID_SEPARATOR = '_'

# What to do if metadata matching fails
FALLBACK_TO_FILENAME_IF_NO_METADATA = True  # Use filename-based batch_id as fallback

# =============================================================================
# CELL ID CONFIGURATION
# =============================================================================

# Assign unique cell_id to ALL cells (recommended)
ASSIGN_UNIVERSAL_CELL_ID = True
CELL_ID_COLUMN = 'cell_id'

# =============================================================================
# MARKER SELECTION CONFIGURATION
# =============================================================================

# Marker selection strategy:
# 'auto_all' - Use all detected potential markers
# 'manual' - Use only markers specified below
MARKER_SELECTION = 'manual'

# Manual marker list
MARKERS_TO_CORRECT = [
    "DAPI", "CD44", "CD31", "CD107a", "E-cadherin", "CD20", "CD4", "CD45RO",
    "Pan-Cytokeratin", "CD68", "Podoplanin", "HLA-A", "CD11c", "CD8", "HLA-DR",
    "Ki67", "CD3e", "CD69", "CD45", "CD15", "gH2AX", "FoxP3", "CD141",
    "PDGFRa", "aSMA", "CD163", "FN1", "S100A9", "CD206", "CD11b", "CD38",
    "PD-L1", "CD40", "PCNA", "Vimentin", "HE_Red", "HE_Green", "HE_Blue"
]

# =============================================================================
# QUALITY CONTROL PARAMETERS
# =============================================================================

# Enable pre-correction quality control
ENABLE_QC = True

# Outlier detection parameters
QC_PERCENTILE_HIGH = 99.9  # Cells above this percentile are considered high outliers
QC_PERCENTILE_LOW = 0.1    # Cells below this percentile are considered low outliers

# Quality filtering criteria
QC_MAX_OUTLIER_FRACTION = 0.5  # Maximum fraction of markers that can be outliers (0.25 = 50% of selected markers)
QC_FILTER_HIGH_OUTLIERS = True   # Filter cells with too many high outliers
QC_FILTER_LOW_OUTLIERS = True    # Filter cells with too many low outliers

# Area-based filtering (optional)
ENABLE_AREA_FILTER = True
AREA_COLUMN = 'area'  # Column name for cell area
AREA_PERCENTILE_LOW = 0.1   # Remove cells smaller than this percentile
AREA_PERCENTILE_HIGH = 99.9 # Remove cells larger than this percentile

# Generate QC histograms
GENERATE_QC_HISTOGRAMS = True
QC_HISTOGRAM_SUBSET = 10  # Generate histograms for first N markers (set to None for all)

# =============================================================================
# ADDITIONAL FILTERING PARAMETERS
# =============================================================================

# Tissue filtering (from annotation pipeline)
TISSUE_FILTER = True
TISSUE_COLUMN = 'tissue'
TISSUE_VALUE = 1

# Additional filtering columns (add your own criteria)
ADDITIONAL_FILTERS = {}
# Example: ADDITIONAL_FILTERS = {'region_type': ['tumor', 'stroma'], 'annotation_status': [1]}

# =============================================================================
# SOM AND PROCESSING PARAMETERS (BIOLOGICALLY OPTIMIZED)
# =============================================================================

# SOM parameters for clustering - UPDATED for better biological preservation
# Larger grid creates more granular clusters, reducing overcorrection
SOM_GRID_SIZE = (20, 20)  # Increased from (8,8) - creates 400 clusters instead of 64
SOM_ITERATIONS = 75000    # Reduced from 100000 - sufficient for convergence, less overfitting
SOM_RANDOM_SEED = 42
ARCSINH_COFACTOR = 5.0    # Cofactor for ArcSinh pre-transformation

# Batch correction strategy and covariates configuration
# cyCombine approach always uses SOM-clustered correction
COVARIATES = ['tissue_roi']              # Biological covariates to protect (e.g. tissue_roi)
MIN_CELLS_PER_CLUSTER = 10              # Minimum cell count for ComBat per cluster
PARAMETRIC_PRIOR = True                 # Use parametric priors in ComBat (True/False)

# =============================================================================
# BATCH CORRECTION METHOD (MATCHING R cyCombine)
# =============================================================================

# Normalization method - CRITICAL PARAMETER matching R cyCombine's norm_method
# Options:
#   'scale' - Apply per-batch Z-score normalization for SOM clustering (RECOMMENDED for data from same experiment)
#   'rank'  - Apply per-batch rank transformation for SOM clustering (RECOMMENDED for cross-experiment integration)
NORMALIZATION_METHOD = 'scale'  # matches cyCombine norm_method="scale" or "rank"

# Processing parameters
SUBSAMPLE_FOR_PLOTS = 100000  # Number of cells to use for visualization
OVERWRITE_EXISTING = False    # Set to True to reprocess existing files

# Quality control parameters
GENERATE_PLOTS = True
PLOT_DPI = 300

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# MARKER SELECTION AND VALIDATION (INTEGRATED FROM CELL 7)
# =============================================================================

# Apply marker selection strategy using utility function
from spacemap.batch_correction_utils import select_and_validate_markers

# Use potential_markers from data discovery if available, otherwise None
potential_markers_list = potential_markers if 'potential_markers' in locals() else None

# Select and validate final markers
final_markers = select_and_validate_markers(
    marker_selection_strategy=MARKER_SELECTION,
    manual_markers=MARKERS_TO_CORRECT,
    potential_markers=potential_markers_list
)

# =============================================================================
# CONFIGURATION SUMMARY
# =============================================================================

print("="*80)
print("BATCH CORRECTION CONFIGURATION SUMMARY")
print("="*80)
print(f"  Input directory: {INPUT_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  CSV file patterns: {CSV_FILE_PATTERNS}")
print(f"  Batch ID strategy: {BATCH_ID_STRATEGY}")
if BATCH_ID_STRATEGY == 'metadata':
    print(f"    Use metadata: {USE_METADATA}")
    if USE_METADATA:
        print(f"    Metadata file: {METADATA_FILE}")
        print(f"    File prefix column: {METADATA_FILE_PREFIX_COLUMN}")
        if METADATA_BATCH_ID_COLUMN:
            print(f"    Batch ID column: {METADATA_BATCH_ID_COLUMN}")
        else:
            print(f"    Batch ID from: {METADATA_BATCH_ID_COMPONENTS} (joined by '{METADATA_BATCH_ID_SEPARATOR}')")
        print(f"    Fallback to filename: {FALLBACK_TO_FILENAME_IF_NO_METADATA}")
elif BATCH_ID_STRATEGY == 'filename':
    print(f"    Remove suffixes: {FILENAME_SUFFIXES_TO_REMOVE[:3]}{'...' if len(FILENAME_SUFFIXES_TO_REMOVE) > 3 else ''}")
elif BATCH_ID_STRATEGY == 'column':
    print(f"    Batch column: {BATCH_COLUMN_NAME}")
elif BATCH_ID_STRATEGY == 'filename_custom':
    print(f"    Custom pattern: {BATCH_ID_CUSTOM_PATTERN}")
print(f"  Marker selection: {MARKER_SELECTION}")
print(f"  Final markers for correction: {len(final_markers)}")
print(f"  Universal cell ID: {ASSIGN_UNIVERSAL_CELL_ID}")
print(f"  Enable QC: {ENABLE_QC}")
print(f"  Generate plots: {GENERATE_PLOTS}")
print("="*80)

## STEP 2: Data Loading and Validation

Load all CSV files from the annotation pipeline, combine them, and perform initial data validation:

**Process:**
- Discover and load all CSV files from the annotated directory
- Combine datasets while preserving batch information
- Validate data integrity and marker availability
- Assign universal cell IDs for traceability

**Output:** Combined dataset ready for quality control analysis.

In [ ]:
# =============================================================================
# STEP 2: DATA LOADING AND VALIDATION
# =============================================================================

print("Step 2: Loading and validating data...")

# Helper function for extracting batch_id from filename
def extract_batch_id_from_filename_simple(filename, suffixes_to_remove):
    """Extract batch_id by removing the first matching suffix"""
    name_without_csv = filename[:-4] if filename.endswith('.csv') else filename
    
    for suffix in suffixes_to_remove:
        suffix_without_csv = suffix[:-4] if suffix.endswith('.csv') else suffix
        if name_without_csv.endswith(suffix_without_csv):
            return name_without_csv[:-len(suffix_without_csv)]
    
    # If no suffix matches, return the whole filename (without .csv)
    return name_without_csv

# Load metadata file if specified
metadata_df = None
if USE_METADATA and BATCH_ID_STRATEGY == 'metadata':
    if METADATA_FILE.exists():
        metadata_df = pd.read_csv(METADATA_FILE)
        print(f"Loaded metadata with {len(metadata_df)} rows")
        print(f"Metadata columns: {list(metadata_df.columns)}")
        
        # Validate required metadata columns
        if METADATA_FILE_PREFIX_COLUMN not in metadata_df.columns:
            print(f"Error: Column '{METADATA_FILE_PREFIX_COLUMN}' not found in metadata.")
            print(f"Available columns: {list(metadata_df.columns)}")
            if FALLBACK_TO_FILENAME_IF_NO_METADATA:
                print("Falling back to filename-based batch assignment")
                BATCH_ID_STRATEGY = 'filename'  # Switch strategy
            else:
                raise ValueError(f"Required metadata column '{METADATA_FILE_PREFIX_COLUMN}' not found")
        
        elif METADATA_BATCH_ID_COLUMN and METADATA_BATCH_ID_COLUMN not in metadata_df.columns:
            # Check if we can create batch_id from components
            missing_components = [col for col in METADATA_BATCH_ID_COMPONENTS if col not in metadata_df.columns]
            if missing_components:
                print(f"Error: Batch ID column '{METADATA_BATCH_ID_COLUMN}' and components {missing_components} not found")
                if FALLBACK_TO_FILENAME_IF_NO_METADATA:
                    print("Falling back to filename-based batch assignment")
                    BATCH_ID_STRATEGY = 'filename'  # Switch strategy
                else:
                    raise ValueError(f"Cannot create batch_id from metadata")
            else:
                print(f"Column '{METADATA_BATCH_ID_COLUMN}' not found, will create from {METADATA_BATCH_ID_COMPONENTS}")
                METADATA_BATCH_ID_COLUMN = None  # Signal to create from components
    else:
        print(f"Warning: Metadata file not found: {METADATA_FILE}")
        if FALLBACK_TO_FILENAME_IF_NO_METADATA:
            print("Falling back to filename-based batch assignment")
            BATCH_ID_STRATEGY = 'filename'  # Switch strategy
        else:
            raise FileNotFoundError(f"Metadata file not found: {METADATA_FILE}")

# Find CSV files using the pattern list
csv_files = []
for pattern in CSV_FILE_PATTERNS:
    csv_files = list(INPUT_DIR.glob(pattern))
    if csv_files:
        print(f"Found {len(csv_files)} files using pattern: {pattern}")
        break

if not csv_files:
    raise ValueError(f"No CSV files found in {INPUT_DIR} using any of these patterns: {CSV_FILE_PATTERNS}")

# Load all data frames
all_data_frames = []

for i, file in enumerate(csv_files):
    print(f"  Loading {file.name}...")
    try:
        df = pd.read_csv(file, low_memory=False)
        
        # Extract batch_id based on strategy
        if BATCH_ID_STRATEGY == 'metadata' and metadata_df is not None:
            # Extract prefix for metadata matching
            prefix = extract_batch_id_from_filename_simple(file.name, FILENAME_SUFFIXES_TO_REMOVE)
            print(f"    Extracted prefix: '{prefix}'")
            
            # Find matching metadata row
            metadata_row = metadata_df[metadata_df[METADATA_FILE_PREFIX_COLUMN] == prefix]
            if metadata_row.empty:
                if FALLBACK_TO_FILENAME_IF_NO_METADATA:
                    print(f"    No metadata match found, using filename batch_id")
                    batch_id = prefix
                else:
                    print(f"    Error: No metadata found for prefix '{prefix}'")
                    continue
            else:
                if len(metadata_row) > 1:
                    print(f"    Warning: Multiple metadata rows for '{prefix}', using first")
                metadata_row = metadata_row.iloc[0]
                
                # Add all metadata columns to dataframe
                for col in metadata_df.columns:
                    if col != METADATA_FILE_PREFIX_COLUMN:
                        df[col] = metadata_row[col]
                
                # Get or create batch_id
                if METADATA_BATCH_ID_COLUMN and METADATA_BATCH_ID_COLUMN in metadata_row:
                    batch_id = metadata_row[METADATA_BATCH_ID_COLUMN]
                    print(f"    Using metadata batch_id: '{batch_id}'")
                else:
                    # Create from components
                    components = [str(metadata_row[col]) for col in METADATA_BATCH_ID_COMPONENTS]
                    batch_id = METADATA_BATCH_ID_SEPARATOR.join(components)
                    print(f"    Created batch_id: '{batch_id}' from {METADATA_BATCH_ID_COMPONENTS}")
        
        elif BATCH_ID_STRATEGY == 'filename':
            batch_id = extract_batch_id_from_filename_simple(file.name, FILENAME_SUFFIXES_TO_REMOVE)
            print(f"    Filename batch_id: '{batch_id}'")
        
        elif BATCH_ID_STRATEGY == 'filename_custom':
            import re
            match = re.search(BATCH_ID_CUSTOM_PATTERN, file.name)
            if match:
                batch_id = match.group(1)
                print(f"    Custom pattern batch_id: '{batch_id}'")
            else:
                print(f"    Warning: Custom pattern didn't match, using full filename")
                batch_id = file.stem  # filename without extension
        
        elif BATCH_ID_STRATEGY == 'column':
            if BATCH_COLUMN_NAME in df.columns:
                batch_id = df[BATCH_COLUMN_NAME]
                print(f"    Using column '{BATCH_COLUMN_NAME}' as batch_id. Unique values: {batch_id.nunique()}")
            else:
                raise KeyError(f"Batch column '{BATCH_COLUMN_NAME}' not found in {file.name}")

        else:
            raise ValueError(f"Unknown BATCH_ID_STRATEGY: {BATCH_ID_STRATEGY}")
        
        # Add batch information to dataframe
        df['file_prefix'] = prefix if BATCH_ID_STRATEGY == 'metadata' else batch_id
        df['batch_id'] = batch_id
        df['batch'] = file.name  # Keep original filename for reference
        
        all_data_frames.append(df)
        
        if i == 0:
            print(f"    First file shape: {df.shape}")
            print(f"    Columns: {len(df.columns)}")
            if BATCH_ID_STRATEGY == 'metadata' and metadata_df is not None:
                added_cols = [col for col in metadata_df.columns if col != METADATA_FILE_PREFIX_COLUMN]
                if added_cols:
                    print(f"    Added metadata columns: {added_cols}")
            
    except Exception as e:
        print(f"    Error loading {file.name}: {e}")
        continue

if not all_data_frames:
    raise ValueError("No data files could be loaded successfully")

# Assign universal cell IDs to ALL cells
if ASSIGN_UNIVERSAL_CELL_ID:
    print("Assigning universal cell IDs to all cells...")
    all_data_frames, total_cells = assign_universal_cell_ids(all_data_frames, CELL_ID_COLUMN)
    print(f"  Assigned unique cell IDs from 1 to {total_cells}")

# Combine all data
print("Combining data...")
combined_data = pd.concat(all_data_frames, ignore_index=True)

# Determine available markers for correction
if 'final_markers' in locals():
    available_markers = [m for m in final_markers if m in combined_data.columns]
else:
    available_markers = [m for m in MARKERS_TO_CORRECT if m in combined_data.columns]

print(f"Available markers for correction: {len(available_markers)}")
if len(available_markers) != len(final_markers if 'final_markers' in locals() else MARKERS_TO_CORRECT):
    missing = set((final_markers if 'final_markers' in locals() else MARKERS_TO_CORRECT)) - set(available_markers)
    print(f"  Missing markers: {list(missing)}")

if not available_markers:
    raise ValueError("No markers available for correction")

print(f"Combined data shape: {combined_data.shape}")
print(f"Unique batches: {combined_data['batch_id'].nunique()}")
print(f"Batch distribution:")
for batch_id, count in combined_data['batch_id'].value_counts().items():
    print(f"  {batch_id}: {count:,} cells")

if ASSIGN_UNIVERSAL_CELL_ID:
    print(f"Universal cell IDs: {combined_data[CELL_ID_COLUMN].min()} to {combined_data[CELL_ID_COLUMN].max()}")
    # Verify uniqueness
    if combined_data[CELL_ID_COLUMN].nunique() != len(combined_data):
        raise ValueError("Cell IDs are not unique!")
    else:
        print("  Cell ID uniqueness verified")

## STEP 3: Quality Control Analysis & Visualization

Perform comprehensive quality control analysis with integrated custom visualization:

**QC Analysis:**
- Outlier detection using percentile thresholds
- Area-based filtering for cell size validation
- Multi-marker quality assessment

**Integrated Visualization:**
- Custom marker distribution plots with pass/fail status
- Z-score transformations and density overlays
- Statistical summaries for each QC marker

**Output:** Quality-filtered dataset with visual QC validation ready for batch correction.

In [ ]:
# =============================================================================
# STEP 3: QUALITY CONTROL ANALYSIS & VISUALIZATION
# =============================================================================

if ENABLE_QC:
    print("Step 3: Performing quality control analysis and visualization...")
    
    # Use the streamlined QC function from spacemap.utils
    from spacemap.batch_correction_utils import perform_quality_control_analysis
    # QC parameters
    QC_PARAMS = {
        'percentile_high': QC_PERCENTILE_HIGH,
        'percentile_low': QC_PERCENTILE_LOW,
        'max_outlier_fraction': QC_MAX_OUTLIER_FRACTION,
        'filter_high_outliers': QC_FILTER_HIGH_OUTLIERS,
        'filter_low_outliers': QC_FILTER_LOW_OUTLIERS,
        'enable_area_filter': ENABLE_AREA_FILTER,
        'area_column': AREA_COLUMN,
        'area_percentile_low': AREA_PERCENTILE_LOW,
        'area_percentile_high': AREA_PERCENTILE_HIGH,
        'generate_histograms': GENERATE_QC_HISTOGRAMS,
        'histogram_subset': QC_HISTOGRAM_SUBSET,
        'generate_plots': GENERATE_PLOTS,
        'plot_dpi': PLOT_DPI
    }
    
    # Perform QC analysis
    combined_data, qc_filtering_stats = perform_quality_control_analysis(
        combined_data, available_markers, QC_PARAMS, QC_DIR
    )
    
    print(f"Initial QC completed. Ready for batch correction.")
else:
    print("Quality control disabled. All cells marked as quality=1")
    combined_data['quality'] = 1

print(f"Total cells available for batch correction: {(combined_data['quality'] == 1).sum():,}")

# =============================================================================
# CUSTOM QC VISUALIZATION (Integrated into QC Step)
# =============================================================================

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Select specific markers to visualize (add/remove as needed)
QC_MARKERS_TO_VISUALIZE = ["area", "DAPI", "CD45", "Pan-Cytokeratin"]

# Visualization options
USE_LOG_TRANSFORM = False  # Set to True to use log10 transformation on x-axis
USE_Z_SCORE = True         # Set to True to show Z-score, False for raw values
SHOW_DENSITY_PLOT = True   # Set to True to overlay density curves
FIGURE_SIZE = (15, 10)     # Figure size (width, height)
BINS = 50                  # Number of histogram bins

print("\nGenerating custom QC visualization...")
print(f"  Markers to visualize: {QC_MARKERS_TO_VISUALIZE}")

# Check which requested markers are available
available_qc_markers = [m for m in QC_MARKERS_TO_VISUALIZE if m in combined_data.columns]
missing_qc_markers = [m for m in QC_MARKERS_TO_VISUALIZE if m not in combined_data.columns]

if missing_qc_markers:
    print(f"  Warning: Missing markers: {missing_qc_markers}")
    
if available_qc_markers:
    print(f"  Visualizing {len(available_qc_markers)} markers: {available_qc_markers}")
    
    # Create subplots
    n_markers = len(available_qc_markers)
    cols = min(2, n_markers)
    rows = (n_markers + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=FIGURE_SIZE)
    if n_markers == 1:
        axes = [axes]
    elif rows == 1:
        axes = [axes] if cols == 1 else axes
    else:
        axes = axes.flatten()
    
    # Define colors for QC status
    colors = {
        'passed': '#2E86AB',      # Blue for passed QC
        'failed': '#F24236',      # Red for failed QC
        'all': '#A23B72'          # Purple for all cells
    }
    
    for i, marker in enumerate(available_qc_markers):
        ax = axes[i]
        
        # Get marker data
        marker_data = combined_data[marker].copy()
        
        # Remove any infinite or NaN values
        marker_data = marker_data.replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(marker_data) == 0:
            ax.text(0.5, 0.5, f'No valid data for {marker}', 
                   ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{marker} - No Data')
            continue
            
        # Apply log transformation if requested
        if USE_LOG_TRANSFORM:
            # Add small constant to avoid log(0)
            min_positive = marker_data[marker_data > 0].min() if (marker_data > 0).any() else 1e-10
            marker_data = np.log10(marker_data + min_positive)
            xlabel_suffix = " (log10)"
        else:
            xlabel_suffix = ""
            
        # Apply Z-score transformation if requested
        if USE_Z_SCORE:
            scaler = StandardScaler()
            marker_data_scaled = scaler.fit_transform(marker_data.values.reshape(-1, 1)).flatten()
            xlabel_prefix = "Z-score of "
        else:
            marker_data_scaled = marker_data.values
            xlabel_prefix = ""
            
        # Get QC status for each cell
        if 'quality' in combined_data.columns:
            qc_passed = combined_data.loc[marker_data.index, 'quality'] == 1
            qc_failed = combined_data.loc[marker_data.index, 'quality'] == 0
        else:
            # If no quality column, assume all passed
            qc_passed = pd.Series([True] * len(marker_data), index=marker_data.index)
            qc_failed = pd.Series([False] * len(marker_data), index=marker_data.index)
        
        # Calculate dynamic range
        data_min, data_max = np.percentile(marker_data_scaled, [0.1, 99.9])
        range_padding = (data_max - data_min) * 0.1
        x_range = (data_min - range_padding, data_max + range_padding)
        
        # Create histograms
        if SHOW_DENSITY_PLOT:
            # Use density plots
            if qc_failed.any():
                ax.hist(marker_data_scaled[qc_failed], bins=BINS, alpha=0.6, 
                       color=colors['failed'], label=f'Failed QC (n={qc_failed.sum():,})',
                       density=True, range=x_range)
            
            if qc_passed.any():
                ax.hist(marker_data_scaled[qc_passed], bins=BINS, alpha=0.6,
                       color=colors['passed'], label=f'Passed QC (n={qc_passed.sum():,})',
                       density=True, range=x_range)
        else:
            # Use count histograms
            if qc_failed.any():
                ax.hist(marker_data_scaled[qc_failed], bins=BINS, alpha=0.6,
                       color=colors['failed'], label=f'Failed QC (n={qc_failed.sum():,})',
                       range=x_range)
            
            if qc_passed.any():
                ax.hist(marker_data_scaled[qc_passed], bins=BINS, alpha=0.6,
                       color=colors['passed'], label=f'Passed QC (n={qc_passed.sum():,})',
                       range=x_range)
        
        # Add vertical lines for QC thresholds if available
        if USE_Z_SCORE and 'quality' in combined_data.columns:
            # Add standard QC threshold lines (typically ±3 or ±2 standard deviations)
            for threshold in [-3, -2, 2, 3]:
                ax.axvline(threshold, color='red', linestyle='--', alpha=0.5, linewidth=1)
        
        # Formatting
        ax.set_xlabel(f'{xlabel_prefix}{marker}{xlabel_suffix}')
        ax.set_ylabel('Density' if SHOW_DENSITY_PLOT else 'Count')
        ax.set_title(f'{marker} Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_xlim(x_range)
        
        # Add summary statistics text
        stats_text = f'Total: {len(marker_data):,}\n'
        if qc_passed.any():
            stats_text += f'Passed: {qc_passed.sum():,} ({qc_passed.sum()/len(marker_data)*100:.1f}%)\n'
        if qc_failed.any():
            stats_text += f'Failed: {qc_failed.sum():,} ({qc_failed.sum()/len(marker_data)*100:.1f}%)'
            
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
               verticalalignment='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Remove empty subplots
    for j in range(n_markers, len(axes)):
        fig.delaxes(axes[j])
    
    plt.tight_layout()
    plt.show()
    
    # Print QC visualization summary
    print(f"\nQC Visualization Summary:")
    total_cells = len(combined_data)
    if 'quality' in combined_data.columns:
        passed_qc = (combined_data['quality'] == 1).sum()
        failed_qc = (combined_data['quality'] == 0).sum()
        print(f"  Total cells analyzed: {total_cells:,}")
        print(f"  Passed QC: {passed_qc:,} ({passed_qc/total_cells*100:.1f}%)")
        print(f"  Failed QC: {failed_qc:,} ({failed_qc/total_cells*100:.1f}%)")
    else:
        print(f"  Total cells: {total_cells:,} (no QC filtering applied)")
        
    print(f"  Visualization mode: {'Z-score' if USE_Z_SCORE else 'Raw values'}")
    if USE_LOG_TRANSFORM:
        print(f"  X-axis: log10 transformed")
else:
    print("  No valid markers found for visualization")

## STEP 4: Batch Correction & Diagnostic Assessment

Perform batch correction with integrated diagnostic evaluation using direct ComBat approach (aligned with R cyCombine):

**Batch Correction Process:**
1. **Data Preparation** → Filter cells for correction (tissue==1, quality==1)
2. **Direct ComBat Correction** → Apply ComBat to all markers simultaneously (no clustering)
3. **Data Integration** → Merge corrected values with original dataset

**Key Features:**
- **Direct ComBat Approach** → Matches proven R cyCombine workflow
- **No SOM Clustering** → Eliminates artificial "no expression" cells
- **Simultaneous Correction** → All markers corrected together for consistency
- **Publication-Ready** → Aligned with established batch correction standards

**Integrated Diagnostics:**
- UMAP visualization of correction quality
- Correction magnitude analysis and variance preservation metrics
- Visual feedback plots for immediate quality assessment
- Recommendations based on correction statistics

**Output:** Batch-corrected dataset with immediate diagnostic feedback for quality evaluation.

In [ ]:
# =============================================================================
# STEP 4: BATCH CORRECTION & DIAGNOSTIC ASSESSMENT
# =============================================================================

print("Step 4: Performing batch correction with integrated diagnostics...")

# Apply data filters for batch correction
FILTER_CONFIG = {
    'enable_qc': ENABLE_QC,
    'tissue_filter': TISSUE_FILTER,
    'tissue_column': TISSUE_COLUMN,
    'tissue_value': TISSUE_VALUE,
    'additional_filters': {},
    'min_cells_per_batch': 100
}

# Filter data for batch correction
from spacemap.batch_correction_utils import apply_data_filters
filtered_data_for_correction = apply_data_filters(combined_data, FILTER_CONFIG)

# Validate marker availability
from spacemap.batch_correction_utils import validate_markers_availability
final_available_markers = validate_markers_availability(MARKERS_TO_CORRECT, filtered_data_for_correction.columns)

# Initialize batch corrector with DIRECT COMBAT approach (aligned with R cyCombine)
from spacemap.batch_correction_utils import BatchCorrector
corrector = BatchCorrector(
    markers=final_available_markers,
    normalization_method=NORMALIZATION_METHOD,  # Matches R cyCombine norm_method parameter
    som_grid_size=SOM_GRID_SIZE,  # Kept for compatibility but not used in direct ComBat
    som_iterations=SOM_ITERATIONS,  # Kept for compatibility but not used in direct ComBat
    random_seed=SOM_RANDOM_SEED,
    verbose=True,
    arcsinh_cofactor=ARCSINH_COFACTOR,
    mode='som_clustered', # Always cyCombine mode
    covariates=COVARIATES,
    min_cells_per_cluster=MIN_CELLS_PER_CLUSTER,
    parametric=PARAMETRIC_PRIOR
)

# Perform batch correction using SOM-clustered cyCombine ComBat
print(f"Performing cyCombine ComBat batch correction on {len(filtered_data_for_correction):,} cells using {len(final_available_markers)} markers...")
print(f"Normalization method: '{NORMALIZATION_METHOD}' (matching R cyCombine: norm_method='{NORMALIZATION_METHOD}')")
print("This approach matches the proven R cyCombine batch_correct() function:")
print("  1. Pre-normalize with StandardScaler (norm_method='scale')")
print("  2. Apply parametric ComBat (parametric=TRUE)")
print("  3. Inverse transform to original scale")
print("  4. Clip negative values to ensure non-negative range")
corrected_subset = corrector.correct_batch_effects(filtered_data_for_correction)

# Preserve all original cells with corrected values where available
from spacemap.batch_correction_utils import preserve_filtered_cells_in_output
combined_data_corrected = preserve_filtered_cells_in_output(
    combined_data, 
    corrected_subset, 
    final_available_markers, 
    CELL_ID_COLUMN
)

print(f"Direct ComBat batch correction completed!")
print(f"  Total cells in dataset: {len(combined_data_corrected):,}")
print(f"  Cells with corrected values: {corrected_subset.shape[0]:,}")
print(f"  Cells filtered out: {len(combined_data_corrected) - corrected_subset.shape[0]:,}")
print(f"  Corrected markers: {len(final_available_markers)}")

print("Batch correction step completed. Generating immediate diagnostic assessment...")

# =============================================================================
# INTEGRATED POST-CORRECTION DIAGNOSTIC ASSESSMENT
# =============================================================================

print("Generating post-correction diagnostic assessment...")

# CRITICAL FIX: Only analyze cells that actually underwent batch correction
# Extract cells that have corrected marker values for diagnostic analysis
has_corrected_data = pd.Series([True] * len(combined_data_corrected), index=combined_data_corrected.index)

# Check if cells have any corrected marker values
for marker in final_available_markers:
    corrected_col = f"{marker}_corr"
    if corrected_col in combined_data_corrected.columns:
        has_corrected_data = has_corrected_data & combined_data_corrected[corrected_col].notna()
    else:
        has_corrected_data = pd.Series([False] * len(combined_data_corrected), index=combined_data_corrected.index)
        break

# Extract only the cells that underwent batch correction for diagnostic analysis
corrected_cells_for_diagnostics = combined_data_corrected[has_corrected_data].copy()

# =============================================================================
# cyCombine Quality Metrics (EMD and MAD)
# =============================================================================

print("\nCalculating cyCombine quality metrics (EMD Reduction and MAD Score)...")
from spacemap.batch_correction_utils import calculate_emd_reduction, calculate_mad_score

emd = calculate_emd_reduction(filtered_data_for_correction, combined_data_corrected, final_available_markers, batch_col='batch')
mad = calculate_mad_score(filtered_data_for_correction, combined_data_corrected, final_available_markers, batch_col='batch')

print(f"  EMD Reduction: {emd:.4f} (higher is better, 1.0 is perfect)")
print(f"  MAD Score: {mad:.4f} (lower is better, 0.0 is perfect)\n")

## STEP 4B: Diagnostic Visualization & QC Threshold Determination

**Generate comprehensive diagnostic visualizations to help you decide on QC thresholds:**

**Visualizations Generated:**
1. **UMAP Before/After Correction** - See batch mixing quality
2. **Correction Distance Distribution** - Identify overcorrected cells
3. **UMAP with Correction Distance Overlay** - Visualize spatial patterns of correction
4. **Batch-specific UMAP panels** - Check individual batch behavior
5. **Marker Variance Analysis** - Assess preservation of biological signal

**Use these plots to:**
- Determine if UMAP outlier filtering is needed (see outlier clusters)
- Set `UMAP1_MIN/MAX` and `UMAP2_MIN/MAX` thresholds based on visual inspection
- Decide on `OVERCORRECTION_PERCENTILE_THRESHOLD` based on correction distance distribution
- Evaluate overall batch correction quality before proceeding to Step 5

**Action Required:** Review plots and adjust Step 5 parameters if needed before running Step 5.

In [ ]:
# =============================================================================
# STEP 4B: DIAGNOSTIC VISUALIZATION CONFIGURATION
# =============================================================================

# =============================================================================
# VISUALIZATION CONFIGURATION (Customize these parameters)
# =============================================================================

# Maximum number of cells to use for UMAP computation (larger = slower but more accurate)
MAX_CELLS_FOR_UMAP = 50000

# UMAP parameters (adjust for different embedding characteristics)
UMAP_N_NEIGHBORS = 15      # Larger = more global structure, smaller = more local structure
UMAP_MIN_DIST = 0.1        # Minimum distance between points (0.0-0.99)
UMAP_RANDOM_SEED = 42      # For reproducibility

# Figure display settings
FIGURE_WIDTH = 1200        # Width of main plots in pixels
FIGURE_HEIGHT = 500        # Height of main plots in pixels

# Marker distribution histograms (Before/After comparison)
SHOW_MARKER_DISTRIBUTIONS = True   # Generate marker distribution plots

# Which markers to visualize in histograms (None = all markers, or specify list)
# Examples:
#   None - Show all markers
#   ['CD45', 'CD3e', 'CD8', 'CD4'] - Show only these specific markers
MARKERS_TO_VISUALIZE = ['CD45', 'CD3e', 'CD8', 'CD4', 'CD68', 'CD20']  # Change to None for all markers

# Transformation for marker intensities in histograms
# Options: 'log10', 'arcsinh', 'none'
MARKER_TRANSFORMATION = 'arcsinh'

# Number of bins for histograms
HISTOGRAM_BINS = 50

# Additional visualization options
VERBOSE_OUTPUT = True      # Print detailed progress messages

# =============================================================================
# RUN DIAGNOSTIC VISUALIZATION
# =============================================================================

if 'corrected_cells_for_diagnostics' not in locals() or corrected_cells_for_diagnostics.empty:
    print("Error: No corrected data available. Please run Step 4 first.")
else:
    # Import diagnostic visualization function
    from spacemap.batch_correction_utils import generate_diagnostic_visualizations
    
    # Prepare marker lists
    original_markers = [col for col in final_available_markers 
                       if col in corrected_cells_for_diagnostics.columns]
    corrected_markers = [f"{col}_corr" for col in final_available_markers 
                        if f"{col}_corr" in corrected_cells_for_diagnostics.columns]
    
    # Set UMAP parameters
    umap_params = {
        'n_neighbors': UMAP_N_NEIGHBORS,
        'min_dist': UMAP_MIN_DIST,
        'random_state': UMAP_RANDOM_SEED
    }
    
    # Generate all diagnostic visualizations
    diagnostic_results = generate_diagnostic_visualizations(
        corrected_data=corrected_cells_for_diagnostics,
        original_markers=original_markers,
        corrected_markers=corrected_markers,
        batch_col='batch_id',
        max_cells_for_umap=MAX_CELLS_FOR_UMAP,
        umap_params=umap_params,
        figure_size=(FIGURE_WIDTH, FIGURE_HEIGHT),
        show_marker_distributions=SHOW_MARKER_DISTRIBUTIONS,
        markers_to_visualize=MARKERS_TO_VISUALIZE,
        marker_transform=MARKER_TRANSFORMATION,
        histogram_bins=HISTOGRAM_BINS,
        verbose=VERBOSE_OUTPUT,
        cofactor=ARCSINH_COFACTOR
    )
    
    # Store results for later use
    if diagnostic_results is not None:
        umap_data = diagnostic_results['umap_data']
        diagnostic_statistics = diagnostic_results['statistics']
        qc_recommendations = diagnostic_results['recommendations']
        
        # Store UMAP coordinates back to main dataset
        print(f"\nStoring UMAP coordinates in corrected dataset...")
        corrected_cells_for_diagnostics.loc[umap_data.index, 'UMAP1'] = umap_data['UMAP1']
        corrected_cells_for_diagnostics.loc[umap_data.index, 'UMAP2'] = umap_data['UMAP2']
        corrected_cells_for_diagnostics.loc[umap_data.index, 'UMAP1_original'] = umap_data['UMAP1_original']
        corrected_cells_for_diagnostics.loc[umap_data.index, 'UMAP2_original'] = umap_data['UMAP2_original']
        corrected_cells_for_diagnostics.loc[umap_data.index, 'correction_distance'] = umap_data['correction_distance']
        
        # Update combined_data_corrected if it exists
        if 'combined_data_corrected' in locals():
            combined_data_corrected.loc[umap_data.index, 'UMAP1'] = umap_data['UMAP1']
            combined_data_corrected.loc[umap_data.index, 'UMAP2'] = umap_data['UMAP2']
            combined_data_corrected.loc[umap_data.index, 'correction_distance'] = umap_data['correction_distance']
        
        print(f"Diagnostic visualization complete!")
        print(f"\nYou can now review the plots above and adjust Step 5 QC parameters accordingly.")
    else:
        print("Error: Diagnostic visualization failed. Please check error messages above.")

## STEP 5: Post-Correction Quality Control Filtering

Apply post-correction quality control filters to remove overcorrected cells and outliers.

**QC Filters:**
- Overcorrection detection using Mahalanobis distance
- Correction distance filtering for extreme corrections
- UMAP outlier filtering using coordinate thresholds

**Output:** Filtered dataset with QC-passed corrected cells, ready for final assembly.

In [ ]:
# =============================================================================
# STEP 5: POST-CORRECTION QUALITY CONTROL FILTERING
# =============================================================================

print("Step 5: Applying post-correction quality control filters...")

# =============================================================================
# POST-CORRECTION QC PARAMETERS
# =============================================================================

ENABLE_POST_CORRECTION_QC = True
FILTER_OVERCORRECTED_CELLS = True
FILTER_HIGH_CORRECTION_DISTANCE = False
FILTER_UMAP_OUTLIERS = False

OVERCORRECTION_PERCENTILE_THRESHOLD = 99
MAX_CORRECTION_DISTANCE = 10.0
MARKER_SPECIFIC_FILTERING = False
MAX_MARKER_CORRECTION_MAGNITUDE = 3.0

USE_PERCENTILE_UMAP_FILTERING = False
UMAP_PERCENTILE_THRESHOLD = 1.0
USE_ABSOLUTE_UMAP_FILTERING = False
UMAP1_MIN = -6
UMAP1_MAX = 15
UMAP2_MIN = -10
UMAP2_MAX = 15

# Compile QC thresholds
POST_QC_THRESHOLDS = {
    'enable_post_correction_qc': ENABLE_POST_CORRECTION_QC,
    'marker_specific_filtering': MARKER_SPECIFIC_FILTERING,
}

if FILTER_OVERCORRECTED_CELLS:
    POST_QC_THRESHOLDS['filter_overcorrected'] = True
    POST_QC_THRESHOLDS['overcorrection_percentile_threshold'] = OVERCORRECTION_PERCENTILE_THRESHOLD

if FILTER_HIGH_CORRECTION_DISTANCE:
    POST_QC_THRESHOLDS['max_correction_distance'] = MAX_CORRECTION_DISTANCE

if MARKER_SPECIFIC_FILTERING:
    POST_QC_THRESHOLDS['max_marker_correction_magnitude'] = MAX_MARKER_CORRECTION_MAGNITUDE

if FILTER_UMAP_OUTLIERS:
    POST_QC_THRESHOLDS['filter_umap_outliers'] = True
    POST_QC_THRESHOLDS['use_percentile_umap_filtering'] = USE_PERCENTILE_UMAP_FILTERING
    POST_QC_THRESHOLDS['umap_percentile_threshold'] = UMAP_PERCENTILE_THRESHOLD
    POST_QC_THRESHOLDS['use_absolute_umap_filtering'] = USE_ABSOLUTE_UMAP_FILTERING
    if USE_ABSOLUTE_UMAP_FILTERING:
        POST_QC_THRESHOLDS['umap1_min'] = UMAP1_MIN
        POST_QC_THRESHOLDS['umap1_max'] = UMAP1_MAX
        POST_QC_THRESHOLDS['umap2_min'] = UMAP2_MIN
        POST_QC_THRESHOLDS['umap2_max'] = UMAP2_MAX

print(f"Post-correction QC configured:")
print(f"  Overcorrection filtering: {FILTER_OVERCORRECTED_CELLS} ({OVERCORRECTION_PERCENTILE_THRESHOLD}th percentile)")
print(f"  Distance filtering: {FILTER_HIGH_CORRECTION_DISTANCE}")
print(f"  UMAP filtering: {FILTER_UMAP_OUTLIERS}")

# =============================================================================
# APPLY QC FILTERS
# =============================================================================

from spacemap.batch_correction_utils import apply_post_correction_qc_filters, detect_overcorrected_cells

if 'combined_data_corrected' in locals() and ENABLE_POST_CORRECTION_QC:
    has_corrected_data = pd.Series([True] * len(combined_data_corrected), index=combined_data_corrected.index)
    
    for marker in final_available_markers:
        corrected_col = f"{marker}_corr"
        if corrected_col in combined_data_corrected.columns:
            has_corrected_data = has_corrected_data & combined_data_corrected[corrected_col].notna()
        else:
            has_corrected_data = pd.Series([False] * len(combined_data_corrected), index=combined_data_corrected.index)
            break
    
    cells_for_analysis = combined_data_corrected[has_corrected_data].copy()
    
    print(f"Analyzing {len(cells_for_analysis):,} corrected cells")
    
    corrected_cells_with_diagnostics = detect_overcorrected_cells(
        data=cells_for_analysis,
        markers=final_markers,
        batch_col='batch_id'
    )

    filtered_corrected_cells, qc_stats = apply_post_correction_qc_filters(
        data=corrected_cells_with_diagnostics,
        qc_thresholds=POST_QC_THRESHOLDS
    )

    print(f"\nQC Results: {len(filtered_corrected_cells):,} / {len(cells_for_analysis):,} cells retained ({qc_stats['retention_rate']:.1%})")
    
    corrected_cells_after_step5_qc = filtered_corrected_cells.copy()

elif not ENABLE_POST_CORRECTION_QC:
    print("Post-correction QC disabled - using all corrected cells")
    if 'combined_data_corrected' in locals():
        has_corrected_data = pd.Series([True] * len(combined_data_corrected), index=combined_data_corrected.index)
        for marker in final_available_markers:
            corrected_col = f"{marker}_corr"
            if corrected_col in combined_data_corrected.columns:
                has_corrected_data = has_corrected_data & combined_data_corrected[corrected_col].notna()
            else:
                has_corrected_data = pd.Series([False] * len(combined_data_corrected), index=combined_data_corrected.index)
                break
        corrected_cells_after_step5_qc = combined_data_corrected[has_corrected_data].copy()
        print(f"Corrected cells: {len(corrected_cells_after_step5_qc):,}")
    else:
        corrected_cells_after_step5_qc = pd.DataFrame()
else:
    corrected_cells_after_step5_qc = pd.DataFrame()

if 'corrected_cells_after_step5_qc' in locals() and not corrected_cells_after_step5_qc.empty:
    print(f"\nStep 5 complete: {len(corrected_cells_after_step5_qc):,} corrected cells ready for assembly")
else:
    print("\nNo corrected cells available - check previous steps")

## STEP 6: Final Dataset Assembly

Assemble the final comprehensive dataset with QC status tracking.

**Process:**
- Merge QC-passed corrected cells with all original cells
- Add 'corrected' status column (1 = passed all QC, 0 = failed or filtered)
- Preserve all original data with proper metadata

**Output:** Complete dataset ready for export and downstream analysis.

In [ ]:
# =============================================================================
# STEP 6: FINAL DATASET ASSEMBLY & QC STATUS TRACKING
# =============================================================================

print("Step 6: Assembling final dataset with QC tracking...")

MIN_CELLS_PER_BATCH_POST_QC = 100

if 'corrected_cells_after_step5_qc' in locals() and not corrected_cells_after_step5_qc.empty:
    final_corrected_data = combined_data.copy()
    
    # Initialize corrected status column
    final_corrected_data['corrected'] = 0
    
    # Add corrected marker columns (initialize with NaN)
    for marker in final_available_markers:
        corrected_col = f"{marker}_corr"
        if corrected_col not in final_corrected_data.columns:
            final_corrected_data[corrected_col] = np.nan
    
    # Update cells that passed QC with corrected values
    if 'cell_id' in corrected_cells_after_step5_qc.columns and 'cell_id' in final_corrected_data.columns:
        corrected_mask = final_corrected_data['cell_id'].isin(corrected_cells_after_step5_qc['cell_id'])
        
        print(f"Updating {corrected_mask.sum():,} cells with corrected values...")
        
        for marker in final_available_markers:
            corrected_col = f"{marker}_corr"
            if corrected_col in corrected_cells_after_step5_qc.columns:
                cell_id_to_corrected = dict(zip(
                    corrected_cells_after_step5_qc['cell_id'], 
                    corrected_cells_after_step5_qc[corrected_col]
                ))
                final_corrected_data.loc[corrected_mask, corrected_col] = final_corrected_data.loc[corrected_mask, 'cell_id'].map(cell_id_to_corrected)
        
        final_corrected_data.loc[corrected_mask, 'corrected'] = 1
    
    # Summary statistics
    total_cells = len(final_corrected_data)
    corrected_cells = (final_corrected_data['corrected'] == 1).sum()
    not_corrected_cells = (final_corrected_data['corrected'] == 0).sum()
    
    print(f"\nFinal dataset assembled:")
    print(f"  Total cells: {total_cells:,}")
    print(f"  Corrected cells (passed all QC): {corrected_cells:,}")
    print(f"  Non-corrected cells: {not_corrected_cells:,}")
    print(f"  Batches: {final_corrected_data['batch_id'].nunique()}")
    
    # Check batch distribution
    if not corrected_cells_after_step5_qc.empty:
        corrected_batch_dist = corrected_cells_after_step5_qc['batch_id'].value_counts()
        small_batches = corrected_batch_dist[corrected_batch_dist < MIN_CELLS_PER_BATCH_POST_QC]
        
        if len(small_batches) > 0:
            print(f"\nWarning: {len(small_batches)} batches have < {MIN_CELLS_PER_BATCH_POST_QC} corrected cells")
            print(f"  Consider adjusting QC thresholds if needed")
    
    # Calculate retention rate
    original_tissue1 = (combined_data['tissue'] == 1).sum() if 'tissue' in combined_data.columns else len(combined_data)
    retention_rate = corrected_cells / original_tissue1 if original_tissue1 > 0 else 0
    
    print(f"\nRetention rate: {retention_rate:.1%} of tissue==1 cells")
    print(f"\nStep 6 complete - dataset ready for export")

else:
    print("Error: No corrected cells from Step 5")
    final_corrected_data = pd.DataFrame()

## STEP 7: Final Visualization

Generate final UMAP visualization of batch-corrected cells after all quality control steps.

**Visualization:** UMAP plot showing batch integration quality and spatial distribution of QC-passed cells.

In [ ]:
# =============================================================================
# STEP 7: FINAL VISUALIZATION
# =============================================================================

import plotly.express as px
import numpy as np

print("Step 7: Generating final visualization...")

# Extract corrected cells for visualization
has_corrected_data = pd.Series([True] * len(final_corrected_data), index=final_corrected_data.index)

for marker in final_available_markers:
    corrected_col = f"{marker}_corr"
    if corrected_col in final_corrected_data.columns:
        has_corrected_data = has_corrected_data & final_corrected_data[corrected_col].notna()
    else:
        has_corrected_data = pd.Series([False] * len(final_corrected_data), index=final_corrected_data.index)
        break

corrected_cells_for_viz = final_corrected_data[has_corrected_data].copy()

print(f"Visualizing {len(corrected_cells_for_viz):,} corrected cells")

# Find batch column
batch_col = 'batch_id' if 'batch_id' in corrected_cells_for_viz.columns else 'batch'

# Check for existing UMAP coordinates
umap_cols = [col for col in corrected_cells_for_viz.columns if 'UMAP' in col.upper()]

if len(umap_cols) >= 2:
    umap1_col, umap2_col = umap_cols[0], umap_cols[1]
    print(f"Using existing UMAP coordinates: {umap1_col}, {umap2_col}")
else:
    print("Computing UMAP on corrected markers...")
    import umap
    from sklearn.preprocessing import StandardScaler
    
    corrected_marker_cols = [f"{marker}_corr" for marker in final_available_markers]
    available_cols = [col for col in corrected_marker_cols if col in corrected_cells_for_viz.columns]
    
    if not available_cols:
        available_cols = final_available_markers
    
    marker_data = corrected_cells_for_viz[available_cols].copy()
    scaler = StandardScaler()
    marker_data_scaled = scaler.fit_transform(marker_data)
    
    # Use subset for large datasets
    if len(corrected_cells_for_viz) > 50000:
        print(f"Large dataset - using 50k cell subset for UMAP")
        subset_idx = np.random.choice(len(corrected_cells_for_viz), 50000, replace=False)
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        reducer.fit(marker_data_scaled[subset_idx])
        umap_embedding = reducer.transform(marker_data_scaled)
    else:
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
        umap_embedding = reducer.fit_transform(marker_data_scaled)
    
    corrected_cells_for_viz['UMAP1'] = umap_embedding[:, 0]
    corrected_cells_for_viz['UMAP2'] = umap_embedding[:, 1]
    umap1_col, umap2_col = 'UMAP1', 'UMAP2'

# Create UMAP visualization
fig = px.scatter(
    corrected_cells_for_viz,
    x=umap1_col,
    y=umap2_col,
    color=batch_col,
    title='Final UMAP: Batch-Corrected Cells After QC',
    labels={umap1_col: 'UMAP1', umap2_col: 'UMAP2'},
    opacity=0.6
)

fig.update_traces(marker=dict(size=2))
fig.update_layout(width=1000, height=700)
fig.show()

print(f"\nVisualization complete")
print(f"Final dataset: {len(corrected_cells_for_viz):,} corrected cells across {corrected_cells_for_viz[batch_col].nunique()} batches")

## STEP 8: Export Final Results

Save the batch-corrected dataset and processing metadata.

**Export Files:**
- Combined corrected dataset (all cells)
- Individual batch files (corrected cells only)
- Processing summary and parameters

In [ ]:
# =============================================================================
# STEP 8: FINAL DATA EXPORT
# =============================================================================

print("Step 8: Exporting final batch-corrected dataset...")

FINAL_OUTPUT_DIR = OUTPUT_DIR
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save batch-corrected files
from spacemap.batch_correction_utils import save_batch_corrected_files
save_batch_corrected_files(
    final_corrected_data, 
    FINAL_OUTPUT_DIR, 
    final_available_markers, 
    CELL_ID_COLUMN
)

# Create processing summary
processing_summary = {
    'processing_timestamp': pd.Timestamp.now().isoformat(),
    'total_input_files': len(csv_files),
    'total_input_cells': len(combined_data),
    'cells_after_initial_qc': (combined_data['quality'] == 1).sum() if 'quality' in combined_data.columns else len(combined_data),
    'cells_after_batch_correction': len(combined_data_corrected),
    'cells_after_post_correction_qc': len(final_corrected_data),
    'final_retention_rate': len(final_corrected_data) / len(combined_data),
    'unique_batches': final_corrected_data['batch_id'].nunique(),
    'corrected_markers': len(final_available_markers),
    'markers_corrected': final_available_markers,
    'normalization_method': NORMALIZATION_METHOD,
    'post_qc_enabled': ENABLE_POST_CORRECTION_QC,
    'overcorrection_threshold': OVERCORRECTION_PERCENTILE_THRESHOLD
}

# Save processing summary
summary_df = pd.DataFrame([processing_summary])
summary_df.to_csv(FINAL_OUTPUT_DIR / 'batch_correction_summary.csv', index=False)

# Save parameters for reproducibility
import json
params = {
    'normalization_method': NORMALIZATION_METHOD,
    'qc_parameters': {
        'enable_qc': ENABLE_QC, 
        'percentile_high': QC_PERCENTILE_HIGH, 
        'percentile_low': QC_PERCENTILE_LOW
    },
    'post_qc_parameters': {
        'enable_post_qc': ENABLE_POST_CORRECTION_QC, 
        'overcorrection_threshold': OVERCORRECTION_PERCENTILE_THRESHOLD, 
        'max_correction_distance': MAX_CORRECTION_DISTANCE
    },
    'markers_corrected': final_available_markers,
    'processing_timestamp': pd.Timestamp.now().isoformat()
}

with open(FINAL_OUTPUT_DIR / 'processing_parameters.json', 'w') as f:
    json.dump(params, f, indent=2)

# Data integrity check
integrity_ok = all(col in final_corrected_data.columns for col in ['cell_id', 'batch_id', 'quality', 'corrected'])

print(f"\nExport complete:")
print(f"  Input cells: {len(combined_data):,}")
print(f"  Output cells: {len(final_corrected_data):,}")
print(f"  Retention: {len(final_corrected_data)/len(combined_data):.1%}")
print(f"  Batches: {final_corrected_data['batch_id'].nunique()}")
print(f"  Corrected markers: {len(final_available_markers)}")
print(f"  Data integrity: {'OK' if integrity_ok else 'Check warnings'}")

print(f"\nOutput directory: {FINAL_OUTPUT_DIR}")
print(f"  all_samples_combined_corrected.csv")
print(f"  batch_correction_summary.csv")
print(f"  processing_parameters.json")